In [17]:
import numpy as np
import pandas as pd
from typing import Tuple
import sympy as sp
from IPython.display import display, Math

pd.set_option('display.precision', 12)
pd.set_option('display.width', 300)
pd.set_option('display.max_columns', None)

In [18]:
def read_matrix(path: str) -> pd.DataFrame:
    data = []
    with open(path, 'r') as f:
        for lineno, line in enumerate(f, start=1):
            tokens = line.strip().split()
            if not tokens:
                continue
            row = []
            for tok in tokens:
                try:
                    if tok.count('/') == 1:
                        num, den = tok.split('/')
                        val = int(num) / int(den)
                    else:
                        val = float(tok)
                except Exception as e:
                    raise ValueError(f"Error parsing token '{tok}' on line {lineno} in {path}: {e}")
                row.append(val)
            data.append(row)

    if not data:
        return pd.DataFrame()

    ncol = len(data[0])
    if any(len(r) != ncol for r in data):
        lengths = [len(r) for r in data]
        raise ValueError(f"Row-length mismatch in {path}: expected {ncol} columns, got {lengths}")

    return pd.DataFrame(data)


In [19]:
def output_matrix(mat, precision=4, display_obj=True):
    if isinstance(mat, pd.DataFrame):
        mat = mat.values
    if not isinstance(mat, np.ndarray):
        mat = np.array(mat)

    def format_val(val):
        if np.isrealobj(val) or np.isreal(val):
            return f"{np.real(val):.{precision}f}"
        else:
            r = np.real(val)
            i = np.imag(val)
            if abs(i) < 1e-10:
                return f"{r:.{precision}f}"
            sign = "+" if i >= 0 else "-"
            return f"{r:.{precision}f} {sign} {abs(i):.{precision}f}i"

    formatted_mat = np.vectorize(format_val)(mat)
    sp_mat = sp.Matrix(formatted_mat)
    latex_str = sp.latex(sp_mat)
    
    if display_obj:
        display(Math(latex_str))
    return latex_str


# Khai triển kỳ dị SVD

## Thuật toán

**Input**: $A_{m \times n}$

**Output**: Các ma trận $U, \Sigma, V^T$ thỏa mãn $A = U\Sigma V^T$.

1. Kiểm tra nếu $m \le n$:

    1.1. Tính ma trận $M = A \cdot A^T$.

    1.2. Tìm các giá trị riêng $\lambda_1 \ge \lambda_2 \ge \dots \ge \lambda_m$ và các vector riêng trực chuẩn $u_1, u_2, \dots, u_m$ tương ứng của $M$.
    
    1.3. Gán $U = \begin{bmatrix} u_1 & u_2 & \dots & u_m \end{bmatrix}$

    1.4. Tính các giá trị kỳ dị $\sigma_i = \sqrt{\lambda_i}$ với $i = \overline{1,m}$.

    1.5. Khởi tạo ma trận $\Sigma_{m \times n} = \text{diag}(\sigma_1, \sigma_2, \ldots, \sigma_m)$
    
    1.6. Khởi tạo ma trận $V_{n \times n}$. Với mỗi $i$ từ $1 \to m$:

    - Nếu $\sigma_i > 0 \Rightarrow$ Gán cột thứ $i$ của $V$ là $v_i = \dfrac{A^T u_i}{\sigma_i}$.

    - Nếu $\sigma_i = 0 \Rightarrow$ Gán cột thứ $i$ của $V$ là vector $0$.

    1.7. Đặt $r$ là số lượng $\sigma_i > 0$. Thực hiện lần lượt các phép tính và gán:

    - Tính các vector $v_{r+1}, \dots, v_n$ thuộc $\text{ker}(A)$ trực chuẩn với $v_1, \dots, v_r$ bằng Gram-Schmidt.

    - Gán $V = \begin{bmatrix} v_1 & v_2 & \dots & v_n \end{bmatrix}$.

2. Nếu $m > n$:

    2.1. Tính ma trận $N = A^T \cdot A$.

    2.2. Tìm các giá trị riêng $\lambda_1 \ge \lambda_2 \ge \dots \ge \lambda_n$ và các vector riêng trực chuẩn $v_1, v_2, \dots, v_n$ tương ứng của $N$.
    
    2.3. Gán $V = \begin{bmatrix} v_1 & v_2 & \dots & v_n \end{bmatrix}$

    2.4. Tính các giá trị kỳ dị $\sigma_j = \sqrt{\lambda_j}$ với $j = 1 \dots n$.

    2.5. Khởi tạo ma trận $\Sigma_{m \times n} = \text{diag}(\sigma_1, \sigma_2, \ldots, \sigma_n)$
    
    2.6. Khởi tạo ma trận $U_{m \times m}$. Với mỗi $j$ từ $1 \to n$:

    - Nếu $\sigma_j > 0 \Rightarrow$ Gán cột thứ $j$ của $U$ là $u_j = \dfrac{A v_j}{\sigma_j}$.

    - Nếu $\sigma_j = 0 \Rightarrow$ Gán cột thứ $j$ của $U$ là vector $0$.

    2.7. Đặt $r$ là số lượng $\sigma_j > 0$. Thực hiện lần lượt các phép tính và gán:

    - Tính các vector $u_{r+1}, \dots, u_m$ thuộc $\text{ker}(A^T)$ trực chuẩn với $u_1, \dots, u_r$ bằng Gram-Schmidt.

    - Gán $U = \begin{bmatrix} u_1 & u_2 & \dots & u_m \end{bmatrix}$
3. Trả về $U$, $\Sigma$, $V^T$.

# Thuật toán SVD rút gọn (Độc lập)

**Input**: Ma trận $A_{m \times n}$.

**Output**: Khai triển kỳ dị rút gọn $A = U_r \Sigma_r V_r^T$.

1. Kiểm tra nếu $m \le n$:

    1.1. Tính ma trận $M = A \cdot A^T$.

    1.2. Tìm các giá trị riêng dương và sắp xếp giảm dần: $\lambda_1 \ge \lambda_2 \ge \dots \ge \lambda_r > 0$, với $r$ là hạng của $A$. Lấy các vector riêng trực chuẩn $u_1, u_2, \dots, u_r$ tương ứng của $M$.
    
    1.3. Gán $U_r = \begin{bmatrix} u_1 & u_2 & \dots & u_r \end{bmatrix}$.

    1.4. Tính các giá trị kỳ dị $\sigma_i = \sqrt{\lambda_i}$ với $i = \overline{1,r}$.

    1.5. Khởi tạo ma trận đường chéo $\Sigma_{r} = \text{diag}(\sigma_1, \sigma_2, \ldots, \sigma_r)$.
    
    1.6. Khởi tạo ma trận $V_r^T$ kích thước $r \times n$. Với mỗi $i$ từ $1 \to r$:

    - Gán hàng thứ $i$ của $V_r^T$ là $v_i^T = \dfrac{u_i^T A}{\sigma_i}$.

2. Nếu $m > n$:

    2.1. Tính ma trận $N = A^T \cdot A$.

    2.2. Tìm các giá trị riêng dương và sắp xếp giảm dần: $\lambda_1 \ge \lambda_2 \ge \dots \ge \lambda_r > 0$, với $r$ là hạng của $A$. Lấy các vector riêng trực chuẩn $v_1, v_2, \dots, v_r$ tương ứng của $N$.
    
    2.3. Gán $V_r^T = \begin{bmatrix} v_1 & v_2 & \dots & v_r \end{bmatrix}^T$.

    2.4. Tính các giá trị kỳ dị $\sigma_j = \sqrt{\lambda_j}$ với $j = \overline{1,r}$.

    2.5. Khởi tạo ma trận đường chéo $\Sigma_{r} = \text{diag}(\sigma_1, \sigma_2, \ldots, \sigma_r)$.
    
    2.6. Khởi tạo ma trận $U_r$ kích thước $m \times r$. Với mỗi $j$ từ $1 \to r$:

    - Gán cột thứ $j$ của $U_r$ là $u_j = \dfrac{A v_j}{\sigma_j}$.

3. Trả về $U_r$, $\Sigma_r$, $V_r^T$.

# Tính số điều kiện của ma trận

**Input**: Các giá trị kỳ dị $\sigma_i$ của $A$ và hạng $r$.

**Output**: Số điều kiện $\kappa(A)$.

1. Nếu $r > 0$, thực hiện tính toán số điều kiện: $\kappa(A) = \dfrac{\sigma_1}{\sigma_r}$.

2. Nếu $r = 0$, trả về thông báo ma trận không có hạng dương (không xác định số điều kiện).

# Xấp xỉ ma trận (Low-rank Approximation)

**Input**: Các ma trận $U, \Sigma, V^T$, hạng ban đầu $r$ và hạng xấp xỉ $k$.

**Output**: Ma trận xấp xỉ $A_k$.

1. So sánh $k$ và $r$. Nếu $k > r \Rightarrow$ Gán $k = r$.

2. Khởi tạo ma trận $A_k = [0]_{m \times n}$.

3. Lặp $i$ từ $1$ đến $k$:
    - Tính $A_k = A_k + \sigma_i u_i v_i^T$.

4. Tính sai số xấp xỉ theo chuẩn 2:
    - Nếu $k < \text{length}(\Sigma) \Rightarrow$ Trả về sai số $\|A - A_k\|_2 = \sigma_{k+1}$.
    - Trái lại $\Rightarrow$ Trả về sai số $\|A - A_k\|_2 = 0$.


In [20]:
def complete_orthonormal_basis(B: np.ndarray, tol: float = 1e-12) -> np.ndarray:
    """
    Given B (n×k) with orthonormal columns, extend to a full n×n orthonormal basis.
    """
    n, k = B.shape
    Q = np.zeros((n, n))
    Q[:, :k] = B
    for j in range(k, n):
        # start with a random vector
        v = np.random.rand(n)
        # orthonormalize against existing columns
        for i in range(j):
            v -= np.dot(Q[:, i], v) * Q[:, i]
        norm = np.linalg.norm(v)
        # if vector is nearly zero, retry
        while norm < tol:
            v = np.random.rand(n)
            for i in range(j):
                v -= np.dot(Q[:, i], v) * Q[:, i]
            norm = np.linalg.norm(v)
        Q[:, j] = v / float(norm)
    return Q

In [21]:
def compute_svd(A_df: pd.DataFrame, tol: float = 1e-10) -> Tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    """
    Compute the SVD of A (m×n) via the smaller of A·Aᵀ or Aᵀ·A.
    Prints intermediate steps using pandas formatting.

    Returns:
        U_df: m×m DataFrame of left singular vectors
        Sigma_df: m×n DataFrame of singular values on the diagonal
        Vt_df: n×n DataFrame of right singular vectors transposed
    """
    # Convert to NumPy
    A = A_df.values.astype(float)
    m, n = A.shape

    print("--- Ma trận đầu vào A ---")
    output_matrix(A_df)

    # Choose smaller route
    if True:
        print("Trường hợp m <= n, sử dụng M = A · Aᵀ ({}×{})".format(m, m))
        M = A.dot(A.T)
        M_df = pd.DataFrame(M)
        print("Ma trận M:")
        output_matrix(M_df)

        # Eigen-decomposition of M
        eigvals, eigvecs = np.linalg.eigh(M)
        # sort descending
        idx = np.argsort(eigvals)[::-1]
        eigvals = eigvals[idx]
        eigvecs = eigvecs[:, idx]

        # Print eigenpairs
        for i, lam in enumerate(eigvals, start=1):
            print(f"Giá trị riêng λ{i} = {lam}")
            print(f"Vector riêng u{i}:")
            output_matrix(pd.DataFrame(eigvecs[:, i-1]))

        # Singular values
        sigmas = np.sqrt(np.clip(eigvals, 0, None))
        print("Các giá trị kỳ dị:")
        for i, s in enumerate(sigmas, start=1):
            print(f"σ{i} = {s}")

        # Build Σ
        Sigma = np.zeros((m, n))
        for i in range(min(m, n)):
            Sigma[i, i] = sigmas[i]
        Sigma_df = pd.DataFrame(Sigma)
        print("Ma trận Σ:")
        output_matrix(Sigma_df)

        # Compute right singular vectors V
        V = np.zeros((n, n))
        for i in range(min(m, n)):
            if sigmas[i] > tol:
                v = A.T.dot(eigvecs[:, i]) / sigmas[i]
            else:
                v = np.zeros(n)
            V[:, i] = v
            print(f"Vector kỳ dị phải v{i+1}:")
            output_matrix(pd.DataFrame(v))

        # Complete V to full orthonormal basis
        V_complete = complete_orthonormal_basis(V[:, :m], tol)
        Vt_df = pd.DataFrame(V_complete.T)

        # U is full eigenvector matrix of M
        U_complete = eigvecs
        U_df = pd.DataFrame(U_complete)

    return U_df, Sigma_df, Vt_df


In [22]:
def compute_svd_2(A_df: pd.DataFrame, tol: float = 1e-10) -> Tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    """
    Compute the SVD of A (m×n) via the smaller of Aᵀ·A.
    Prints intermediate steps using pandas formatting.

    Returns:
        U_df: m×m DataFrame of left singular vectors
        Sigma_df: m×n DataFrame of singular values on the diagonal
        Vt_df: n×n DataFrame of right singular vectors transposed
    """
    # Convert to NumPy
    A = A_df.values.astype(float)
    m, n = A.shape

    print("--- Ma trận đầu vào A ---")
    output_matrix(A_df)

    # Choose smaller route
    if True:
        print("Trường hợp m > n, sử dụng N = Aᵀ · A ({}×{})".format(n, n))
        N = A.T.dot(A)
        N_df = pd.DataFrame(N)
        print("Ma trận N:")
        output_matrix(N_df)

        # Eigen-decomposition of N
        eigvals, eigvecs = np.linalg.eigh(N)
        idx = np.argsort(eigvals)[::-1]
        eigvals = eigvals[idx]
        eigvecs = eigvecs[:, idx]

        for j, lam in enumerate(eigvals, start=1):
            print(f"Giá trị riêng λ{j} = {lam}")
            print(f"Vector riêng v{j}:")
            output_matrix(pd.DataFrame(eigvecs[:, j-1]))

        sigmas = np.sqrt(np.clip(eigvals, 0, None))
        print("Các giá trị kỳ dị:")
        for j, s in enumerate(sigmas, start=1):
            print(f"σ{j} = {s}")

        Sigma = np.zeros((m, n))
        for j in range(min(m, n)):
            Sigma[j, j] = sigmas[j]
        Sigma_df = pd.DataFrame(Sigma)
        print("Ma trận Σ:")
        output_matrix(Sigma_df)

        # Compute left singular vectors U
        U = np.zeros((m, m))
        for j in range(min(m, n)):
            if sigmas[j] > tol:
                u = A.dot(eigvecs[:, j]) / sigmas[j]
            else:
                u = np.zeros(m)
            U[:, j] = u
            print(f"Vector kỳ dị trái u{j+1}:")
            output_matrix(pd.DataFrame(u))

        # Complete U to full orthonormal basis
        U_complete = complete_orthonormal_basis(U[:, :n], tol)
        U_df = pd.DataFrame(U_complete)

        # V is full eigenvector matrix of N
        V_complete = eigvecs
        Vt_df = pd.DataFrame(V_complete.T)

    return U_df, Sigma_df, Vt_df


In [23]:
def compute_reduced_svd(A_df, tol=1e-10):
    A = A_df.values.astype(float)
    m, n = A.shape
    
    print("\n--- TÍNH SVD RÚT GỌN ĐỘC LẬP ---")
    
    if m <= n:
        T = A.dot(A.T)
        print(f"Trường hợp m <= n, sử dụng T = A · Aᵀ ({m}x{m})")
        eigvals, eigvecs = np.linalg.eigh(T)
        idx = np.argsort(eigvals)[::-1]
        eigvals = eigvals[idx]
        eigvecs = eigvecs[:, idx]
        
        r = np.sum(eigvals > tol)
        print(f"Hạng của ma trận A: r = {r}")
        
        sigmas = np.sqrt(np.clip(eigvals[:r], 0, None))
        
        U_r = eigvecs[:, :r]
        Sigma_r = np.diag(sigmas)
        
        Vt_r = np.zeros((r, n))
        for i in range(r):
            Vt_r[i, :] = A.T.dot(U_r[:, i]) / sigmas[i]
            
    else:
        T = A.T.dot(A)
        print(f"Trường hợp m > n, sử dụng T = Aᵀ · A ({n}x{n})")
        eigvals, eigvecs = np.linalg.eigh(T)
        idx = np.argsort(eigvals)[::-1]
        eigvals = eigvals[idx]
        eigvecs = eigvecs[:, idx]
        
        r = np.sum(eigvals > tol)
        print(f"Hạng của ma trận A: r = {r}")
        
        sigmas = np.sqrt(np.clip(eigvals[:r], 0, None))
        
        Vt_r = eigvecs[:, :r].T
        Sigma_r = np.diag(sigmas)
        
        U_r = np.zeros((m, r))
        for i in range(r):
            U_r[:, i] = A.dot(Vt_r[i, :].T) / sigmas[i]

    print("Ma trận U_r:")
    output_matrix(U_r)
    print("Ma trận Σ_r:")
    output_matrix(Sigma_r)
    print("Ma trận V_r^T:")
    output_matrix(Vt_r)
    
    import pandas as pd
    return pd.DataFrame(U_r), pd.DataFrame(Sigma_r), pd.DataFrame(Vt_r)


In [24]:
def analyze_svd(A, U, S, Vt, k=None, tol=1e-10, precision=4):
    if hasattr(A, 'values'): A = A.values.astype(float)
    if hasattr(U, 'values'): U = U.values.astype(float)
    if hasattr(S, 'values'): S = S.values.astype(float)
    if hasattr(Vt, 'values'): Vt = Vt.values.astype(float)
    
    m, n = A.shape
    sigmas = np.diag(S)
    r = np.sum(sigmas > tol)
    
    print(f"\n--- 1. TÍNH SỐ ĐIỀU KIỆN ---")
    if r > 0:
        cond_num = sigmas[0] / sigmas[r-1]
        print(f"Số điều kiện κ(A) = σ_1 / σ_r = {sigmas[0]:.{precision}f} / {sigmas[r-1]:.{precision}f} = {cond_num:.{precision}f}")
    else:
        print("Ma trận 0.")
        
    print(f"\n--- 2. XẤP XỈ MA TRẬN (LOW-RANK APPROXIMATION) ---")
    if k is not None:
        if k > r:
            k = r
        print(f"Xấp xỉ ma trận A với k = {k}:")
        Ak = np.zeros_like(A)
        for i in range(k):
            u_i = U[:, i].reshape(-1, 1)
            vt_i = Vt[i, :].reshape(1, -1)
            Ak += sigmas[i] * u_i.dot(vt_i)
        print(f"Ma trận A_{k}:")
        output_matrix(Ak, precision)
        
        if k < len(sigmas):
            print(f"Sai số xấp xỉ chuẩn 2 = σ_{k+1} = {sigmas[k]:.{precision}f}")
        else:
            print("Sai số xấp xỉ chuẩn 2 = 0")


In [25]:
A_df = read_matrix("SVD_input_A.txt")

print("\n===========================")
print("1. TÍNH SVD ĐẦY ĐỦ (FULL SVD)")
print("===========================")
U, S, Vt = compute_svd(A_df, tol=1e-9)

print("--- Ma trận A ---")
output_matrix(A_df)
print("--- Ma trận U ---")
output_matrix(U)
print("--- Ma trận Σ ---")
output_matrix(S)
print("--- Ma trận V^T ---")
output_matrix(Vt)

print("\n===========================")
print("2. TÍNH SVD RÚT GỌN ĐỘC LẬP")
print("===========================")
Ur, Sr, Vtr = compute_reduced_svd(A_df, tol=1e-9)

print("\n===========================")
print("3. CÁC PHÂN TÍCH KHÁC (Số đ.kiện, Xấp xỉ)")
print("===========================")
analyze_svd(A_df, U, S, Vt, k=2, precision=4)



1. TÍNH SVD ĐẦY ĐỦ (FULL SVD)
--- Ma trận đầu vào A ---


<IPython.core.display.Math object>

Trường hợp m <= n, sử dụng M = A · Aᵀ (4×4)
Ma trận M:


<IPython.core.display.Math object>

Giá trị riêng λ1 = 345.18097878658665
Vector riêng u1:


...

Ma trận V_r^T:


<IPython.core.display.Math object>


3. CÁC PHÂN TÍCH KHÁC (Số đ.kiện, Xấp xỉ)

--- 1. TÍNH SỐ ĐIỀU KIỆN ---
Số điều kiện κ(A) = σ_1 / σ_r = 18.5790 / 2.1021 = 8.8382

--- 2. XẤP XỈ MA TRẬN (LOW-RANK APPROXIMATION) ---
Xấp xỉ ma trận A với k = 2:
Ma trận A_2:


<IPython.core.display.Math object>

Sai số xấp xỉ chuẩn 2 = σ_3 = 2.1021
